In [ ]:
import os
import pandas as pd

In [ ]:
# Read the CSV file
file_path = r"\\pn.vai.org\projects\pospisilik\vari-core-generated-data\OIC\Ilaria\OIC-168_p53HFD_PaperRevisions\p53\p53_Detection_Level_Quants.csv"
df = pd.read_csv(file_path)

# Get the list of columns that start with 'CD45:'
cd45_columns = [col for col in df.columns if col.startswith('CD45:')]

# Create a dictionary to map old column names to new column names
rename_dict = {col: col.replace('CD45:', 'P53:') for col in cd45_columns}

# Rename the columns
df = df.rename(columns=rename_dict)

# Get all columns that start with 'P53:' (including both original and renamed)
p53_columns = [col for col in df.columns if col.startswith('P53:')]

# Create a dictionary to store the groups of matching columns
column_groups = {}

# Group columns by their exact names
for col in p53_columns:
    matching_cols = [c for c in p53_columns if c == col]
    if len(matching_cols) > 1:
        column_groups[col] = matching_cols

# Merge matching columns by combining their values row-wise
for col_name, cols_to_merge in column_groups.items():
    # Get all non-null values for each row across matching columns
    merged_values = df[cols_to_merge].values.tolist()
    merged_series = pd.Series([
        next((val for val in row if pd.notna(val)), pd.NA)
        for row in merged_values
    ])
    
    # Drop the original columns
    df = df.drop(columns=cols_to_merge)
    
    # Add the merged column back
    df[col_name] = merged_series

# Display the first few rows to verify the changes
print("\nLast few rows of the DataFrame with merged columns:")
print(df.tail())


# Print summary of merges
if column_groups:
    print("\nSummary of merged columns:")
    for col, merged_cols in column_groups.items():
        print(f"- Merged {len(merged_cols)} instances of '{col}'")
else:
    print("\nNo matching columns were found to merge.")

In [ ]:
# Save the modified DataFrame back to a new CSV file
output_path = os.path.join(os.path.dirname(file_path), 'p53_Detection_Level_Quants_merged.csv')
df.to_csv(output_path, index=False)
print(f"\nSaved merged data to: {output_path}")